# Noise models

反復測定から観測分散を推定する `ReplicateNoiseSingleTaskGP` を使い、latent function の不確かさと測定ノイズを分離します。


## このNotebookで学ぶこと

- **潜在関数の不確実性**と**観測ノイズ**の違い
- 同一条件の反復測定からノイズ分散を推定する方法
- 入力に応じてノイズが変化するheteroskedastic GPの考え方
- Mixed・MultiTask・MultiFidelityへの展開
- 有限候補集合での候補提案と、その限界

製造工程では同じ設定を繰り返しても測定値が変わります。ノイズを単なるモデル誤差として扱わず、観測機構の性質として捉えることが本Notebookの目的です。

## 反復測定データ


### 潜在関数と測定ノイズ

観測値を $y(x)=f(x)+\varepsilon(x)$ と書くと、$f(x)$ は知りたい真の応答、$\varepsilon(x)$ は測定ノイズです。

- **潜在関数の事後分散**：学習データから推定した $f(x)$ 自体の不確かさ
- **観測ノイズ分散**：同じ $x$ を繰り返し測定したときのばらつき

両者は異なります。測定ノイズが大きいからといって、潜在関数の不確実性と同じ値になるわけではありません。

In [ ]:
import torch
from robotorchan.models import ReplicateNoiseSingleTaskGP

torch.manual_seed(0)
torch.set_default_dtype(torch.double)
base_X = torch.linspace(0, 1, 9).unsqueeze(-1)
n_repeats = 8
noise_std = 0.05
X = base_X.repeat_interleave(n_repeats, dim=0)
Y = torch.sin(2 * torch.pi * X) + noise_std * torch.randn_like(X)


9種類の入力条件をそれぞれ8回測定します（72観測）。旧例の0、0.5、1はすべて正弦波の零点であり、反復を増やしても山・谷の情報を得られません。今回は0.25と0.75も含め、関数形状を観測します。測定ノイズの標準偏差は0.05です。

## 反復を集約してGPを構築


In [ ]:
model = ReplicateNoiseSingleTaskGP.from_replicates(X, Y)
model.fit()
print(model.raw_train_X)
print(model.raw_train_Yvar)
print(model.replicate_counts)


### `from_replicates()`で何が起こるか

`ReplicateNoiseSingleTaskGP.from_replicates(X, Y)` は同一入力条件の反復観測をまとめ、観測値と反復測定由来の分散情報を使ってGPを構築します。

`raw_train_X`、`raw_train_Yvar`、`replicate_counts` を確認してください。**反復のばらつきの推定値**と**平均値の推定精度**は、反復数に依存して異なるため、分散の定義を混同しないことが重要です。

各条件の不偏標本分散を $s_i^2$、反復数を $r_i$ とすると、集約平均の観測分散は $s_i^2/r_i$（下限値適用前）です。`replicate_variance` と `raw_train_Yvar` は元の応答単位の二乗であり、単発測定と集約平均の区別が重要です。Exact GP の `fit()` は周辺対数尤度を最適化し、変分GPのエポック数とは異なります。

## Posterior


In [ ]:
test_X = torch.linspace(0, 1, 5).unsqueeze(-1)
posterior = model.posterior(test_X)
print(posterior.mean.squeeze(-1))

assert torch.isfinite(posterior.mean).all()
assert (posterior.mean.max() - posterior.mean.min()).item() > 0.5


### 予測平均と潜在関数の区間

`posterior()` の平均と分散を使い、1次元の予測曲線と近似95%の潜在関数区間を描画します。図の区間は**測定値が95%の確率で入る区間ではありません**。

9種類の入力条件で測定した結果と真値を比較します。

In [ ]:
import matplotlib.pyplot as plt

plot_X = torch.linspace(0.0, 1.0, 101).unsqueeze(-1)
model.eval()
with torch.no_grad():
    plot_post = model.posterior(plot_X)
    plot_mean = plot_post.mean.squeeze(-1)
    plot_std = plot_post.variance.clamp_min(0).sqrt().squeeze(-1)
fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(X.squeeze(-1), Y.squeeze(-1), s=15, alpha=0.45, label="replicate observations")
ax.errorbar(model.raw_train_X[:, 0], model.raw_train_Y[:, 0],
            yerr=model.raw_train_Yvar[:, 0].sqrt(), fmt="o", capsize=3,
            label="group mean ± standard error")
ax.plot(plot_X.squeeze(-1), torch.sin(2 * torch.pi * plot_X).squeeze(-1),
        "--", label="true latent function")
ax.plot(plot_X.squeeze(-1), plot_mean, label="posterior mean")
ax.fill_between(plot_X.squeeze(-1), plot_mean - 1.96 * plot_std,
                plot_mean + 1.96 * plot_std, alpha=0.2,
                label="approx. 95% latent interval")
ax.set_xlabel("x")
ax.set_ylabel("response")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()
print('Grid RMSE:', (plot_mean - torch.sin(2 * torch.pi * plot_X[:, 0])).square().mean().sqrt().item())

### 反復測定の分散と平均の分散

左は各条件の反復観測と集約平均、右は単発測定の不偏分散と平均の推定分散です。破線は生成時の既知の分散です。少数の反復から求めた分散には推定誤差があります。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
gx = model.raw_train_X[:, 0]
gy = model.raw_train_Y[:, 0]
s2 = model.replicate_variance[:, 0]
mean_var = model.raw_train_Yvar[:, 0]
counts = model.replicate_counts
torch.testing.assert_close(mean_var, (s2 / counts).clamp_min(1e-6))
axes[0].scatter(X[:, 0], Y[:, 0], alpha=0.45, s=16, label="Raw replicates")
axes[0].errorbar(gx, gy, yerr=s2.sqrt(), fmt="o", capsize=3, label="Mean ± sample SD")
axes[0].errorbar(gx, gy, yerr=mean_var.sqrt(), fmt="s", capsize=3, label="Mean ± SE")
axes[0].set_ylabel("Response")
axes[1].plot(gx, s2, "o-", label="Replicate variance")
axes[1].plot(gx, mean_var, "s-", label="Mean variance")
axes[1].axhline(noise_std**2, linestyle="--", label="True observation variance")
axes[1].axhline(noise_std**2 / n_repeats, linestyle=":", label="True mean variance")
axes[1].set_ylabel("Variance (response squared)")
for ax in axes:
    ax.set_xlabel("x")
    ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

## Replicate-aware candidate proposal

反復観測から推定したノイズ分散を持つモデルで、予測平均最大の候補を選択します。


In [ ]:
candidate_grid = torch.linspace(0, 1, 101).unsqueeze(-1)
model.eval()
with torch.no_grad():
    scores = model.posterior(candidate_grid).mean.squeeze(-1)
candidate = candidate_grid[scores.argmax().item():scores.argmax().item() + 1]
candidate_Y = torch.sin(2 * torch.pi * candidate)
assert candidate.shape == (1, 1)
print("candidate:", candidate, "evaluated objective:", candidate_Y)


### 候補提案の位置付け

既存コードでは101点の候補から**事後平均が最大の点**を選択します。これはgreedyな活用であり、EI・NEIなどの獲得関数最適化ではありません。

測定ノイズが大きい場合は、未観測点を試すだけでなく、既存条件で**追加の反復測定**を行う価値があることもあります。再測定と新規探索の選択は、この単純な予測平均ランキングには含まれません。

### 反復回数と測定平均の精度

同じ測定ノイズを仮定した場合、反復回数が増えるほど平均の標準誤差は $1/\sqrt{r}$ に比例して減少します。測定そのもののノイズが小さくなるわけではありません。

In [ ]:
repeat_grid = torch.arange(2, 33)
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(repeat_grid, noise_std / repeat_grid.sqrt(), label="Theoretical mean SE")
ax.axhline(noise_std, linestyle="--", label="Single measurement SD")
ax.scatter(counts, mean_var.sqrt(), alpha=0.6, label="Empirical mean SE")
ax.set_xlabel("Replicates per condition")
ax.set_ylabel("Standard deviation / standard error")
ax.legend()
fig.tight_layout()
plt.show()

## Mixed replicate-aware candidate proposal

連続値とカテゴリ値の組合せごとに反復測定を集約し、元の混合入力空間で予測平均最大の候補を選びます。これは獲得関数の最適化ではありません。


In [ ]:
from robotorchan.models import MixedReplicateNoiseSingleTaskGP

mixed_base_X = torch.cartesian_prod(base_X[:, 0], torch.tensor([0.0, 1.0]))
mixed_replicate_X = mixed_base_X.repeat_interleave(n_repeats, dim=0)
mixed_replicate_Y = (
    torch.sin(2 * torch.pi * mixed_replicate_X[:, :1])
    + 0.2 * mixed_replicate_X[:, 1:2]
    + 0.03 * torch.randn(mixed_replicate_X.shape[0], 1)
)
mixed_model = MixedReplicateNoiseSingleTaskGP.from_replicates(
    mixed_replicate_X, mixed_replicate_Y, cat_dims=[1]
)
mixed_model.fit()
mixed_pool = torch.cartesian_prod(
    torch.linspace(0, 1, 21), torch.tensor([0.0, 1.0])
)
mixed_model.eval()
with torch.no_grad():
    mixed_scores = mixed_model.posterior(mixed_pool).mean.squeeze(-1)
mixed_candidate = mixed_pool[mixed_scores.argmax()].unsqueeze(0)
mixed_observation = (
    torch.sin(2 * torch.pi * mixed_candidate[:, :1])
    + 0.2 * mixed_candidate[:, 1:2]
)
assert mixed_candidate.shape == (1, 2)
assert mixed_candidate[0, 1].item() in (0.0, 1.0)
print("mixed candidate:", mixed_candidate)
print("evaluated objective:", mixed_observation)


### Mixed入力での反復測定

Mixedモデルでは、連続入力とカテゴリ入力の**組合せが同一**の場合に反復測定を集約します。例えば `(x=0.5, category=0)` と `(x=0.5, category=1)` は異なる条件です。

`cat_dims=[1]` は2列目がカテゴリである指定です。候補プールではカテゴリ0と1を列挙し、連続変数と組み合わせて評価します。

### Mixed版：カテゴリごとの予測

カテゴリ別に集約平均、潜在関数の区間、真値を比較します。星は有限候補集合から選んだ候補の予測平均です。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for category, ax in enumerate(axes):
    query = torch.cat([plot_X, torch.full_like(plot_X, float(category))], dim=-1)
    with torch.no_grad():
        post = mixed_model.posterior(query)
        mean = post.mean.reshape(-1)
        std = post.variance.clamp_min(0).sqrt().reshape(-1)
    mask = mixed_model.raw_train_X[:, 1] == category
    raw_mask = mixed_replicate_X[:, 1] == category
    ax.scatter(mixed_replicate_X[raw_mask, 0], mixed_replicate_Y[raw_mask, 0],
               alpha=0.35, s=12, label="Replicates")
    ax.errorbar(mixed_model.raw_train_X[mask, 0], mixed_model.raw_train_Y[mask, 0],
                yerr=mixed_model.raw_train_Yvar[mask, 0].sqrt(), fmt="o", label="Mean ± SE")
    ax.plot(plot_X[:, 0], torch.sin(2 * torch.pi * plot_X[:, 0]) + 0.2 * category,
            "--", label="Truth")
    ax.plot(plot_X[:, 0], mean, label="Posterior mean")
    ax.fill_between(plot_X[:, 0], mean - 1.96 * std, mean + 1.96 * std, alpha=0.2)
    if int(mixed_candidate[0, 1]) == category:
        ax.scatter(mixed_candidate[0, 0], mixed_scores.max(), marker="*", s=120, label="Candidate")
    ax.set_title(f"Category {category}")
    ax.set_xlabel("x")
    ax.legend(fontsize=7)
axes[0].set_ylabel("Response")
fig.tight_layout()
plt.show()

## Input-dependent noise: single-task GPs

既存の `fit(iterations=1)` で平均モデルと対数ノイズモデルを交互学習します。平均のPosteriorと予測観測分散を分けて確認し、有限プールから候補を選びます。


In [ ]:
from robotorchan.models import (
    HeteroskedasticSingleTaskGP,
    MixedHeteroskedasticSingleTaskGP,
)

hetero_base_X = torch.linspace(0.05, 0.95, 10).unsqueeze(-1)
hetero_repeats = 8
hetero_X = hetero_base_X.repeat_interleave(hetero_repeats, dim=0)
hetero_Y = (
    torch.sin(2 * torch.pi * hetero_X)
    + 0.03 * (1 + hetero_X) * torch.randn_like(hetero_X)
)
hetero_model = HeteroskedasticSingleTaskGP(hetero_X, hetero_Y)
hetero_model.fit(iterations=1)
hetero_model.eval()
hetero_pool = torch.linspace(0, 1, 31).unsqueeze(-1)
with torch.no_grad():
    hetero_scores = hetero_model.posterior(hetero_pool).mean.squeeze(-1)
    hetero_noise = hetero_model.predicted_noise(hetero_pool)
hetero_candidate = hetero_pool[hetero_scores.argmax()].unsqueeze(0)
assert torch.isfinite(hetero_noise).all()
assert hetero_noise.max() > 2 * hetero_model.noise_floor
print('Estimated noise SD range:', hetero_noise.sqrt().min().item(), hetero_noise.sqrt().max().item())
print("heteroskedastic candidate:", hetero_candidate)
print("objective:", torch.sin(2 * torch.pi * hetero_candidate))

mixed_hetero_X = torch.cat(
    [hetero_X, torch.arange(10).remainder(2).repeat_interleave(hetero_repeats).to(hetero_X).unsqueeze(-1)],
    dim=-1,
)
mixed_hetero_Y = hetero_Y + 0.2 * mixed_hetero_X[:, 1:2]
mixed_hetero_model = MixedHeteroskedasticSingleTaskGP(
    mixed_hetero_X, mixed_hetero_Y, cat_dims=[1]
)
mixed_hetero_model.fit(iterations=1)
mixed_hetero_model.eval()
with torch.no_grad():
    mixed_hetero_scores = (
        mixed_hetero_model.posterior(mixed_pool).mean.squeeze(-1)
    )
    mixed_hetero_noise = mixed_hetero_model.predicted_noise(mixed_pool)
mixed_hetero_candidate = mixed_pool[mixed_hetero_scores.argmax()].unsqueeze(0)
assert torch.isfinite(mixed_hetero_noise).all()
print("mixed heteroskedastic candidate:", mixed_hetero_candidate)
print(
    "objective:",
    torch.sin(2 * torch.pi * mixed_hetero_candidate[:, :1])
    + 0.2 * mixed_hetero_candidate[:, 1:2],
)


### 入力依存ノイズ（heteroskedasticity）

反復測定モデルとは別に、**ノイズの大きさが入力条件によって変化する**問題を扱います。例えば高温条件でセンサノイズが増える場合などです。

`fit(iterations=1)` は平均モデルとノイズモデルを交互に学習する既存APIです。反復測定に基づく分散を使うため、ここでは1回のノイズGP更新と最終応答GPの再学習を実行します。**1回の更新は収束やノイズ推定精度を保証しません**。

### 入力依存ノイズの予測

潜在関数の区間と入力依存の観測標準偏差を別々に描きます。各条件を8回測定し、反復分散からノイズGPを学習します。推定値は有限回の反復に依存し、真値との一致は保証されません。

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7), sharex=True, sharey="row")
for col, (name, surrogate, category) in enumerate([
    ("Continuous", hetero_model, None),
    ("Mixed category 0", mixed_hetero_model, 0),
    ("Mixed category 1", mixed_hetero_model, 1),
]):
    query = plot_X if category is None else torch.cat(
        [plot_X, torch.full_like(plot_X, float(category))], dim=-1
    )
    offset = 0 if category is None else 0.2 * category
    with torch.no_grad():
        post = surrogate.posterior(query)
        mean = post.mean.reshape(-1)
        std = post.variance.clamp_min(0).sqrt().reshape(-1)
        noise_std_pred = surrogate.predicted_noise(query).reshape(-1).clamp_min(0).sqrt()
    axes[0, col].plot(plot_X[:, 0], torch.sin(2 * torch.pi * plot_X[:, 0]) + offset,
                      "--", label="Truth")
    axes[0, col].plot(plot_X[:, 0], mean, label="Mean")
    axes[0, col].fill_between(plot_X[:, 0], mean - 1.96 * std, mean + 1.96 * std, alpha=0.2)
    axes[1, col].plot(plot_X[:, 0], noise_std_pred, label="Estimated observation SD")
    axes[1, col].plot(plot_X[:, 0], 0.03 * (1 + plot_X[:, 0]), "--", label="True observation SD")
    axes[0, col].set_title(name)
    axes[1, col].set_xlabel("x")
    for ax in axes[:, col]:
        ax.legend(fontsize=7)
axes[0, 0].set_ylabel("Response")
axes[1, 0].set_ylabel("Observation SD")
fig.tight_layout()
plt.show()

## Heteroskedastic multitask and multi-fidelity GPs

Long-formatのタスク列と忠実度列を区別して入力依存ノイズを学習します。候補は対象タスクまたは高忠実度に固定した有限プールの予測平均から選択します。


In [ ]:
from robotorchan.models import (
    HeteroskedasticMultiFidelityGP,
    HeteroskedasticMultiTaskGP,
    MixedHeteroskedasticMultiTaskGP,
)

task0_X = torch.cat([hetero_X, torch.zeros_like(hetero_X)], dim=-1)
task1_X = torch.cat([hetero_X, torch.ones_like(hetero_X)], dim=-1)
hetero_task_X = torch.cat([task0_X, task1_X], dim=0)
hetero_task_Y = torch.cat([
    hetero_Y,
    torch.sin(2 * torch.pi * hetero_X) + 0.2
    + 0.03 * (1 + hetero_X) * torch.randn_like(hetero_X),
], dim=0)
hetero_task_pool = torch.cat(
    [hetero_pool, torch.zeros_like(hetero_pool)], dim=-1
)
task_noise_model = HeteroskedasticMultiTaskGP(
    hetero_task_X, hetero_task_Y, task_feature=1
)
task_noise_model.fit(iterations=1)
task_noise_model.eval()
with torch.no_grad():
    task_scores = task_noise_model.posterior(hetero_task_pool).mean.reshape(-1)
    task_noise = task_noise_model.predicted_noise(hetero_task_pool)
task_candidate = hetero_task_pool[task_scores.argmax()].unsqueeze(0)
assert torch.isfinite(task_noise).all()
print("task candidate:", task_candidate)
print("task 0 objective:", torch.sin(2 * torch.pi * task_candidate[:, :1]))

mixed_task_X = torch.cat(
    [mixed_hetero_X, torch.zeros(mixed_hetero_X.shape[0], 1)], dim=-1
)
mixed_task_X = torch.cat(
    [
        mixed_task_X,
        torch.cat(
            [mixed_hetero_X, torch.ones(mixed_hetero_X.shape[0], 1)], dim=-1
        ),
    ],
    dim=0,
)
mixed_task_Y = torch.cat([
    mixed_hetero_Y,
    torch.sin(2 * torch.pi * hetero_X) + 0.2 * mixed_hetero_X[:, 1:2] + 0.2
    + 0.03 * (1 + hetero_X) * torch.randn_like(hetero_X),
], dim=0)
mixed_task_pool = torch.cat(
    [mixed_pool, torch.zeros(mixed_pool.shape[0], 1)], dim=-1
)
mixed_task_noise_model = MixedHeteroskedasticMultiTaskGP(
    mixed_task_X, mixed_task_Y, task_feature=2, cat_dims=[1]
)
mixed_task_noise_model.fit(iterations=1)
mixed_task_noise_model.eval()
with torch.no_grad():
    mixed_task_scores = (
        mixed_task_noise_model.posterior(mixed_task_pool).mean.reshape(-1)
    )
    mixed_task_noise = mixed_task_noise_model.predicted_noise(mixed_task_pool)
mixed_task_candidate = mixed_task_pool[mixed_task_scores.argmax()].unsqueeze(0)
assert torch.isfinite(mixed_task_noise).all()
print("mixed task candidate:", mixed_task_candidate)
print(
    "mixed task 0 objective:",
    torch.sin(2 * torch.pi * mixed_task_candidate[:, :1])
    + 0.2 * mixed_task_candidate[:, 1:2],
)

fidelity_X = torch.cat(
    [hetero_X, torch.arange(10).remainder(2).repeat_interleave(hetero_repeats).to(hetero_X).unsqueeze(-1)],
    dim=-1,
)
fidelity_Y = hetero_Y + 0.15 * (1 - fidelity_X[:, 1:2])
fidelity_noise_model = HeteroskedasticMultiFidelityGP(
    fidelity_X, fidelity_Y, data_fidelities=[1]
)
fidelity_noise_model.fit(iterations=1)
fidelity_noise_model.eval()
fidelity_pool = torch.cat(
    [hetero_pool, torch.ones_like(hetero_pool)], dim=-1
)
with torch.no_grad():
    fidelity_scores = fidelity_noise_model.posterior(fidelity_pool).mean.reshape(-1)
    fidelity_noise = fidelity_noise_model.predicted_noise(fidelity_pool)
fidelity_candidate = fidelity_pool[fidelity_scores.argmax()].unsqueeze(0)
assert torch.isfinite(fidelity_noise).all()
print("high-fidelity candidate:", fidelity_candidate)
print("high-fidelity objective:", torch.sin(2 * torch.pi * fidelity_candidate[:, :1]))


### MultiTaskとMultiFidelityのノイズ

- **MultiTask**：製品・設備・測定対象などのタスクによってノイズの性質が異なる可能性があります
- **MultiFidelity**：忠実度（簡易シミュレーションと実測など）によってノイズの性質が異なる可能性があります

task IDとfidelityはどちらも入力列に含まれますが、意味は異なります。以下の例では目的タスクまたは高忠実度を固定して候補を選びます。

### タスク・忠実度ごとの平均と観測ノイズ

各タスクまたは忠実度を固定して予測します。Mixed版はカテゴリ0に固定します。真の観測標準偏差は共通の生成式に基づきます。

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7), sharex=True, sharey="row")
for col, (name, surrogate, mixed, fidelity) in enumerate([
    ("MultiTask", task_noise_model, False, False),
    ("Mixed MultiTask (category 0)", mixed_task_noise_model, True, False),
    ("MultiFidelity", fidelity_noise_model, False, True),
]):
    for level in (0, 1):
        columns = [plot_X]
        if mixed:
            columns.append(torch.zeros_like(plot_X))
        columns.append(torch.full_like(plot_X, float(level)))
        query = torch.cat(columns, dim=-1)
        offset = 0.15 * (1 - level) if fidelity else 0.2 * level
        with torch.no_grad():
            mean = surrogate.posterior(query).mean.reshape(-1)
            noise = surrogate.predicted_noise(query).reshape(-1).clamp_min(0).sqrt()
        line, = axes[0, col].plot(plot_X[:, 0], mean, label=f"Level {level}: mean")
        axes[0, col].plot(plot_X[:, 0], torch.sin(2 * torch.pi * plot_X[:, 0]) + offset,
                          "--", color=line.get_color(), label=f"Level {level}: truth")
        axes[1, col].plot(plot_X[:, 0], noise, label=f"Level {level}: estimated SD")
    axes[1, col].plot(plot_X[:, 0], 0.03 * (1 + plot_X[:, 0]), "--", label="True SD")
    axes[0, col].set_title(name)
    axes[1, col].set_xlabel("x")
    for ax in axes[:, col]:
        ax.legend(fontsize=7)
axes[0, 0].set_ylabel("Response")
axes[1, 0].set_ylabel("Observation SD")
fig.tight_layout()
plt.show()

## Replicate-noise multi-fidelity candidate proposal

忠実度を含む同一条件の反復測定を集約し、最高忠実度に固定した候補を予測平均で選びます。


In [ ]:
from robotorchan.models import ReplicateNoiseMultiFidelityGP

replicate_fidelity_X = fidelity_X
replicate_fidelity_Y = (
    torch.sin(2 * torch.pi * replicate_fidelity_X[:, :1])
    + 0.15 * (1 - replicate_fidelity_X[:, 1:2])
    + 0.03 * torch.randn(replicate_fidelity_X.shape[0], 1)
)
replicate_fidelity_model = ReplicateNoiseMultiFidelityGP.from_replicates(
    replicate_fidelity_X, replicate_fidelity_Y, data_fidelities=[1]
)
from botorch.fit import fit_gpytorch_mll

fit_gpytorch_mll(replicate_fidelity_model.make_mll())
replicate_fidelity_model.eval()
with torch.no_grad():
    replicate_fidelity_scores = (
        replicate_fidelity_model.posterior(fidelity_pool).mean.reshape(-1)
    )
replicate_fidelity_candidate = (
    fidelity_pool[replicate_fidelity_scores.argmax()].unsqueeze(0)
)
assert replicate_fidelity_candidate.shape == (1, 2)
assert replicate_fidelity_candidate[0, 1].item() == 1.0
print("replicate multi-fidelity candidate:", replicate_fidelity_candidate)
print(
    "high-fidelity objective:",
    torch.sin(2 * torch.pi * replicate_fidelity_candidate[:, :1]),
)


### MultiFidelityでの反復測定

同じ設計変数でも、忠実度が違えば**同一測定条件の反復**とは限りません。`data_fidelities=[1]` は2列目を忠実度として扱う指定です。

候補提案では忠実度を1.0に固定しています。これは高忠実度での目的値を求める例であり、測定コストを含めたcost-aware最適化ではありません。

### 忠実度別の集約観測と予測

忠実度0と1を固定した予測曲線と観測平均を比較します。誤差棒は集約平均の標準誤差です。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for fidelity, ax in enumerate(axes):
    query = torch.cat([plot_X, torch.full_like(plot_X, float(fidelity))], dim=-1)
    with torch.no_grad():
        post = replicate_fidelity_model.posterior(query)
        mean = post.mean.reshape(-1)
        std = post.variance.clamp_min(0).sqrt().reshape(-1)
    mask = replicate_fidelity_model.raw_train_X[:, 1] == fidelity
    ax.errorbar(replicate_fidelity_model.raw_train_X[mask, 0],
                replicate_fidelity_model.raw_train_Y[mask, 0],
                yerr=replicate_fidelity_model.raw_train_Yvar[mask, 0].sqrt(),
                fmt="o", capsize=3, label="Mean ± SE")
    ax.plot(plot_X[:, 0], torch.sin(2 * torch.pi * plot_X[:, 0]) + 0.15 * (1 - fidelity),
            "--", label="Truth")
    ax.plot(plot_X[:, 0], mean, label="Mean")
    ax.fill_between(plot_X[:, 0], mean - 1.96 * std, mean + 1.96 * std, alpha=0.2)
    ax.set_title(f"Fidelity {fidelity}")
    ax.set_xlabel("x")
    ax.legend(fontsize=8)
axes[0].set_ylabel("Response")
fig.tight_layout()
plt.show()

## Joint variational heteroskedastic GP

平均と対数観測分散を同時に変分学習する2モデルです。既存の `training_loss()` をAdamで最小化し、候補は有限プールの予測平均から選びます。獲得関数最適化ではありません。


In [ ]:
from robotorchan.models import (
    JointHeteroskedasticSingleTaskGP,
    MixedJointHeteroskedasticSingleTaskGP,
)

joint_variants = [
    (
        "JointHeteroskedasticSingleTaskGP",
        JointHeteroskedasticSingleTaskGP(
            hetero_X, hetero_Y, num_inducing=8, num_mc_samples=32
        ),
        hetero_pool,
    ),
    (
        "MixedJointHeteroskedasticSingleTaskGP",
        MixedJointHeteroskedasticSingleTaskGP(
            mixed_hetero_X, mixed_hetero_Y,
            cat_dims=[1], num_inducing=8, num_mc_samples=32,
        ),
        mixed_pool,
    ),
]
for name, joint_model, pool in joint_variants:
    joint_model.train()
    joint_optimizer = torch.optim.Adam(joint_model.parameters(), lr=0.01)
    for _ in range(200):
        joint_optimizer.zero_grad()
        joint_loss = joint_model.training_loss()
        joint_loss.backward()
        joint_optimizer.step()
    joint_model.eval()
    with torch.no_grad():
        joint_scores = joint_model.posterior(pool).mean.reshape(-1)
        joint_noise = joint_model.predicted_noise(pool)
    joint_candidate = pool[joint_scores.argmax()].unsqueeze(0)
    assert torch.isfinite(joint_scores).all()
    assert torch.isfinite(joint_noise).all()
    if joint_candidate.shape[-1] == 1:
        joint_objective = torch.sin(2 * torch.pi * joint_candidate)
    else:
        joint_objective = (
            torch.sin(2 * torch.pi * joint_candidate[:, :1])
            + 0.2 * joint_candidate[:, 1:2]
        )
    print(name, "candidate:", joint_candidate)
    print(name, "evaluated objective:", joint_objective)


### Joint variational heteroskedastic GP

平均と対数観測分散を同時に変分学習する方法です。`num_inducing` は誘導点数、`num_mc_samples` は近似に使うMonte Carloサンプル数に関する設定です。

ここではAdamを200ステップ実行します。学習回数だけで収束が保証されるわけではなく、実運用では収束確認と、予測平均・ノイズ分散の校正評価が必要です。

### Joint変分GP：平均とノイズの同時学習

平均とノイズの推定を別々に確認します。予測ノイズは分散として返されるため、平方根を取って生成時の標準偏差と比較します。

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7), sharex=True, sharey="row")
for col, (name, surrogate, category) in enumerate([
    ("Joint continuous", joint_variants[0][1], None),
    ("Joint mixed 0", joint_variants[1][1], 0),
    ("Joint mixed 1", joint_variants[1][1], 1),
]):
    query = plot_X if category is None else torch.cat(
        [plot_X, torch.full_like(plot_X, float(category))], dim=-1
    )
    offset = 0 if category is None else 0.2 * category
    with torch.no_grad():
        post = surrogate.posterior(query)
        mean = post.mean.reshape(-1)
        std = post.variance.clamp_min(0).sqrt().reshape(-1)
        noise = surrogate.predicted_noise(query).reshape(-1).clamp_min(0).sqrt()
    axes[0, col].plot(plot_X[:, 0], torch.sin(2 * torch.pi * plot_X[:, 0]) + offset,
                      "--", label="Truth")
    axes[0, col].plot(plot_X[:, 0], mean, label="Posterior mean")
    axes[0, col].fill_between(plot_X[:, 0], mean - 1.96 * std, mean + 1.96 * std, alpha=0.2)
    axes[1, col].plot(plot_X[:, 0], noise, label="Estimated observation SD")
    axes[1, col].plot(plot_X[:, 0], 0.03 * (1 + plot_X[:, 0]), "--", label="True observation SD")
    axes[0, col].set_title(name)
    axes[1, col].set_xlabel("x")
    for ax in axes[:, col]:
        ax.legend(fontsize=7)
axes[0, 0].set_ylabel("Response")
axes[1, 0].set_ylabel("Observation SD")
fig.tight_layout()
plt.show()

## まとめ：ノイズモデルの使い分け

| データの特徴 | 候補となるモデル | 確認事項 |
|---|---|---|
| 同じ条件の反復測定がある | ReplicateNoise GP | 反復数と集約分散の意味 |
| 条件によって測定ノイズが変わる | Heteroskedastic GP | 平均とノイズの識別・学習収束 |
| 平均とノイズを同時に近似学習したい | Joint variational heteroskedastic GP | 誘導点・MCサンプル数・収束 |
| タスクや忠実度も変化する | MultiTask / MultiFidelity版 | task列・fidelity列の対応 |

**観測ノイズが大きいこと**と**目的関数が不確かであること**は別です。どちらを減らしたいかによって、再測定・探索・モデル選択の方針が変わります。